In [164]:
# Install required libs
!pip install -q torch torchvision datasets pillow tqdm

**Liabrary** **And** **Load Data**

In [165]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from datasets import load_dataset
from PIL import Image
from tqdm import tqdm
import string


dataset = load_dataset("Teklia/IAM-line")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['image', 'text'],
        num_rows: 6482
    })
    validation: Dataset({
        features: ['image', 'text'],
        num_rows: 976
    })
    test: Dataset({
        features: ['image', 'text'],
        num_rows: 2915
    })
})


**Vocabulary**

In [166]:

all_text = "".join(dataset["train"]["text"])

chars = sorted(list(set(all_text)))
blank = "_"

alphabet = chars + [blank]

vocab = {c:i for i,c in enumerate(alphabet)}
inv_vocab = {i:c for c,i in vocab.items()}
num_classes = len(vocab)

print("Vocab size:", num_classes)

Vocab size: 80


**Transform**

In [167]:
transform = transforms.Compose([
    transforms.Resize((32, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

**Custom** **Dataset** **Wrapper**

In [168]:
class IAMDataset(Dataset):
    def __init__(self, hf_dataset, vocab, transform=None):
        self.data = hf_dataset
        self.vocab = vocab
        self.transform = transform

    def encode_text(self, text):
        return torch.tensor([self.vocab[c] for c in text], dtype=torch.long)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        img = item["image"].convert("L")
        text = item["text"]

        if self.transform:
            img = self.transform(img)

        label = self.encode_text(text)

        return img, label, text

**CTC**

In [169]:
def ctc_collate(batch):
    imgs = [item[0] for item in batch]
    labels = [item[1] for item in batch]

    imgs = torch.stack(imgs)

    flat_labels = torch.cat(labels)
    label_lengths = torch.tensor([len(l) for l in labels])
    input_lengths = torch.full(size=(len(batch),), fill_value=imgs.shape[-1] // 4, dtype=torch.long)

    return imgs, flat_labels, input_lengths, label_lengths

**Create** **Train** **Loader**

In [170]:

train_data = dataset["train"].select(range(2000))

train_dataset = IAMDataset(train_data, vocab, transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=ctc_collate,
    num_workers=2,
    pin_memory=True
)

**CRNN** **Model**

In [171]:
class CRNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d((2,1)),

            nn.Conv2d(256, 512, 3, padding=1), nn.ReLU(),
            nn.BatchNorm2d(512),
            nn.MaxPool2d((2,1)),
        )

        self.lstm = nn.LSTM(512, 256, 2, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        feat = self.encoder(x)
        b, c, h, w = feat.size()

        # feat = feat.squeeze(2)
        feat = torch.mean(feat, dim=2)
        feat = feat.permute(0, 2, 1)

        out, _ = self.lstm(feat)
        out = self.fc(out)

        return out

**Training** **Setup**

In [172]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = CRNN(num_classes).to(device)

criterion = nn.CTCLoss(blank=vocab[blank], zero_infinity=True)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

**Training** **Loop**

In [173]:
epochs = 3

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for imgs, labels, in_len, lab_len in tqdm(train_loader):
        imgs = imgs.to(device)
        labels = labels.to(device)
        in_len = in_len.to(device)
        lab_len = lab_len.to(device)

        preds = model(imgs)
        preds = preds.permute(1,0,2)

        loss = criterion(preds, labels, in_len, lab_len)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}: Loss {total_loss:.4f}")


100%|██████████| 250/250 [00:11<00:00, 21.79it/s]


Epoch 1: Loss 53.9813


100%|██████████| 250/250 [00:07<00:00, 32.29it/s]


Epoch 2: Loss 73.9471


100%|██████████| 250/250 [00:05<00:00, 45.16it/s]

Epoch 3: Loss 74.8735


**Decoder**

In [174]:
def decode(preds):
    out = preds.argmax(2)
    results = []

    for seq in out:
        text = ""
        prev = -1
        for idx in seq:
            idx = idx.item()
            if idx != prev and inv_vocab[idx] != "_":
                text += inv_vocab[idx]
            prev = idx
        results.append(text)

    return results

**Save** **Predictions**

In [175]:
model.eval()

with open("model_outputs.txt", "w", encoding="utf-8") as f:
    count = 0

    for img, _, text in train_dataset:
        img = img.unsqueeze(0).to(device)

        with torch.no_grad():
            pred = model(img)
            pred = pred.permute(1,0,2)
            pred_text = decode(pred)[0]

        f.write(f"GT: {text} | Pred: {pred_text}\n")

        count += 1
        if count >= 10:
            break

print("Saved model_outputs.txt")

with open("model_outputs.txt", "r") as f:
    print(f.read())

Saved model_outputs.txt
GT: put down a resolution on the subject | Pred: 
GT: and he is to be backed by Mr. Will | Pred: 
GT: nominating any more Labour life Peers | Pred: 
GT: M Ps tomorrow. Mr. Michael Foot has | Pred: 
GT: Griffiths, M P for Manchester Exchange . | Pred: 
GT: is to be made at a meeting of Labour | Pred: 
GT: A MOVE to stop Mr. Gaitskell from | Pred: 
GT: 0M P for Manchester Exchange . | Pred: 
GT: A MOVE to stop Mr. Gaitskell from nominating | Pred: 
GT: meeting of Labour 0M Ps tommorow . Mr. Michael | Pred: 



**Save** **Model**

In [176]:
torch.save(model.state_dict(), "line_ocr_model.pth")
print("Model saved!")



Model saved!
